![](https://github.com/destination-earth/DestinE-DataLake-Lab/blob/main/img/DestinE-banner.jpg?raw=true)

# Extraction of ClimateDT data
Data portfolio: https://confluence.ecmwf.int/display/DDCZ/Climate+DT+Phase+1+data+catalogue#ClimateDTPhase1datacatalogue-Fieldsonasinglelevelorsurface


Requirements:

- params: 228/167
- temporal extent: 1990-2020
- spatial: Austria

- variables
  - temperature
    - only specific time stamps
  - precipitation
    - houerly data needed  

based on these we can only extract data from IFS-NEMO CMIP6 corresponding to scenario index 0

In [ ]:
# all imnports needed to run the notebook
from typing import Literal, Dict
import pandas as pd
from pathlib import Path

import earthkit.geo.cartography

**The following dependencies are missing in the current kernel**

In [ ]:
pip install -q polytope-client covjsonkit joblib

In [ ]:
%%capture cap
%run ./src/desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]
access_token

In [ ]:
scenarios = pd.read_csv("./dta_hydro_dt_exp.csv", sep=";")
scenarios

**Only scenario with index 0 will be considered hereafter, IFS-NEMO | sfc | 1990-2024 | 167/228**

In [ ]:
scenario = scenarios.iloc[0]

In [ ]:
def get_request_dict(experiment: str,
                      activity: str,
                      level_type: str,
                      datestring: str,
                      model: str,
                      parameter: str,
                      location: list,
                      feature: Literal["timeseries", "polygon"]="timeseries",
                      time_resolution: str="0000/to/2300",
                      resolution: str="high") -> Dict:
    # request time-series data
    if feature == "timeseries":
        feature_dict = {
            "type" : "timeseries",
            "points": location,
            "time_axis": "date"
        }
    elif feature == "polygon":
        feature_dict = {
            "type" : "polygon",
            "shape": location
        }
    else:
        raise TypeError("feature not supported")
    
    request = {
        # static parameters of climate dt data
        "class": "d1",
        "dataset": "climate-dt",
        "generation": "1",
        "expver": "0001",
        "stream": "clte",
        "type": "fc",
        # generic
        "activity": activity,
        "experiment": experiment,
        "levtype": level_type,
        "date": datestring,
        "model": model,
        "param": parameter,
        "realization": "1",
        "resolution": resolution,
        "time": time_resolution,
        "feature": feature_dict
    }

    # commented out to check if only one level is request it gets faster or not
    if level_type == "sol":
        # request["levelist"] = "1/to/5"
        request["levelist"] = "1"

    # return the request dict
    return request    
    

In [ ]:
def extract_dtdata(request_dict: dict,
                   bucketname: str="",
                   dt_url: str="polytope.lumi.apps.dte.destination-earth.eu"):
    import earthkit.data as ekd
   
    try:
        # get covjson
        ds = ekd.from_source("polytope", 
                             "destination-earth", 
                             request_dict,
                             stream=False, 
                             address=dt_url)
    except Exception as e:
        print(e)        
        
    return ds.to_xarray()

## Get polygon for Austria

In [ ]:
countries = ["Austria"]
polygon_at = earthkit.geo.cartography.country_polygons(countries, resolution=50e6)

## Extract TS of data over Austria and the given scenario

In [ ]:
#Supress default INFO logging

import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [ ]:
def extract_austria_ts(scenario: dict, datestr: str, params: str, polygon: list):
    request_dict = get_request_dict(scenario["experiment"],
                        scenario["activity"],
                        scenario["level type"],
                        datestr,
                        scenario["model"],
                        params,
                        polygon,
                        "polygon")
    return extract_dtdata(request_dict)

Extract one day of data to get all points within Austria

In [ ]:
ds = extract_austria_ts(scenario, "19900101", params="167/228", polygon=polygon_at)

In [ ]:
npoints_austria = ds["points"].size

In [ ]:
from datetime import datetime
start, end = scenario["temporal extent"].split("-")
ndays = (datetime(int(end)+1,1,1) - datetime(int(start),1,1)).days
ntimestamps = 24 * ndays
ntimestamps

## Write zarr store which gets finally populated with data

In [ ]:
import pandas as pd

# Define start and end dates
start = "1990-01-01"
end = "2025-12-01"

# Generate monthly start dates
dates = pd.date_range(start=start, end=end, freq='MS')  # 'MS' = Month Start

# Build series of formatted date strings
date_strings = pd.Series([
    f"{d.strftime('%Y%m%d')}/to/{(d + pd.offsets.MonthBegin(1)).strftime('%Y%m%d')}"
    for d in dates
])

print(date_strings)

In [ ]:
import earthkit
n_down_threads = 10
earthkit.data.config.set("number-of-download-threads", n_down_threads)
earthkit.data.config

In [ ]:
import s3fs
from rich.prompt import Prompt

bucket = "68e13833a1624f43ba2cac01376a18af:destine-climate-dt"
prefix = ""
s3_endpoint = "https://objects.eodc.eu"

# Create the S3FileSystem with a custom endpoint
s3_eodc = s3fs.S3FileSystem(
    endpoint_url=s3_endpoint,
    key=Prompt.ask(prompt="S3 Key"),
    secret=Prompt.ask(prompt="S3 Secret", password=True),
    use_ssl=True,
)

In [ ]:
s3_eodc.ls("destine-climate-dt")

In [ ]:
s3_eodc.mkdir("destine-climate-dt/IFS-NEMO-down")

In [ ]:
import concurrent.futures
import xarray as xr
import os

with concurrent.futures.ThreadPoolExecutor(max_workers=n_down_threads) as executor:
    futures = []
    for ind, datestr in date_strings[0:2].items():
        futures.append(executor.submit(extract_austria_ts, 
                                       scenario=scenario, 
                                       datestr=datestr,
                                       params="167/228",
                                       polygon=polygon_at))

    counter=0
    for future in concurrent.futures.as_completed(futures):
        future.result().squeeze().chunk(
            chunks={"datetimes": -1, "points": -1}).to_zarr(
                store=s3_eodc.get_mapper(f"destine-climate-dt/IFS-NEMO-down/IFS-NEMO-CMIP_part{counter}.zarr"),mode="w")
        counter += 1